# Imports

In [4]:
import albumentations as A
from torch.utils.data import DataLoader
from segmentation_models_pytorch.losses import DiceLoss, SoftBCEWithLogitsLoss, JaccardLoss, FocalLoss

# Custom Library
import ActivationPrototypes_SARSeg.thesis_utils as utils
from ActivationPrototypes_SARSeg.thesis_utils import *

initialized = False

# Setup

In [5]:
from importlib import reload
reload(utils)
from ActivationPrototypes_SARSeg.thesis_utils import *

In [6]:
lmdb_path = "F:\\Thesis\\Datasets\\Big Earth\\Encoded-BigEarthNet"
parquet_path = "F:\\Thesis\Datasets\\Big Earth\\metadata.parquet"

In [7]:
if not initialized:
    train_matches, val_matches, test_matches = match_keys(parquet_path)

In [19]:
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
],is_check_shapes=False)

print(f"Train size: {len(train_matches)}, Val size: {len(val_matches)}, Test size: {len(test_matches)}")

# Create datasets	
train_dataset = SARSegmentationDataset120(lmdb_path, train_matches[:], transform=transform)
val_dataset = SARSegmentationDataset120(lmdb_path, val_matches[:], transform=None)
test_dataset = SARSegmentationDataset120(lmdb_path, test_matches[:], transform=None)

# Create data loaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

#Define the model metrcis and load model. 
num_train_img = len(train_dataset)
num_val_img = len(val_dataset)
num_test_img = len(test_dataset)

steps_per_epoch = num_train_img//batch_size
val_steps_per_epoch = num_val_img//batch_size
print("Steps per epoch: ", steps_per_epoch)
print("Validation steps per epoch: ", val_steps_per_epoch)

Train size: 237871, Val size: 122342, Test size: 119825
Opening LMDB environment ...
Opening LMDB environment ...
Opening LMDB environment ...
Steps per epoch:  14866
Validation steps per epoch:  7646


In [20]:
random_indices_train = random.sample(range(len(train_matches)), 10000)
random_indices_val = random.sample(range(len(val_matches)), 4000)
random_indices_test = random.sample(range(len(test_matches)), 4000)

random_train_matches = [train_matches[i] for i in random_indices_train]
random_val_matches = [val_matches[i] for i in random_indices_val]
random_test_matches = [test_matches[i] for i in random_indices_test]

train_dataset_short = SARSegmentationDataset120(lmdb_path, random_train_matches[:], transform=transform)
val_dataset_short = SARSegmentationDataset120(lmdb_path, random_val_matches[:], transform=None)
test_dataset_short = SARSegmentationDataset120(lmdb_path, random_test_matches[:], transform=None)

train_loader_short = DataLoader(train_dataset_short, batch_size=batch_size, shuffle=True)
val_loader_short = DataLoader(val_dataset_short, batch_size=batch_size, shuffle=False)
test_loader_short = DataLoader(test_dataset_short, batch_size=batch_size, shuffle=False)

Opening LMDB environment ...
Opening LMDB environment ...
Opening LMDB environment ...


# Tensorflow Board

In [9]:
# %load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir=runs --port=6010

Reusing TensorBoard on port 6010 (pid 30612), started 18:45:10 ago. (Use '!kill 30612' to kill it.)

# Training

In [21]:
want_to_train120 = True

model = load_base_with_bigearth_pretrained120()
# model = load_from_checkpoint120("../models/unet120_epoch_1.pth")

# Freeze encoder layers
for param in model.encoder.parameters():
    param.requires_grad = False
    
if want_to_train120:
    criterion_base = FocalLoss(mode='multiclass', ignore_index=20)  
    # model = create_base_model()
    training(
        model = model, 
        epoch_start = 1,
        epoch_end =  10,
        loss_fn = criterion_base,
        train_loader = train_loader_short,
        val_loader = val_loader_short,
        num_classes = 20,
        model_name = "unet120_10k",)
    # for param in model.encoder.parameters():
    #     param.requires_grad = False

else:
    print("Not training")

c:\Users\Jean\anaconda3\envs\ActivationPrototypes_SARSeg\lib\site-packages\configilm\ConfigILM.py:134: UserWarning: Keyword 'img_size' unknown. Trying to ignore and restart creation.
  warnings.warn(f"Keyword '{failed_kw}' unknown. Trying to ignore and restart creation.")


Training unet120_10k from epoch 1 to 10


Epoch 1/10:   0%|          | 0/625 [00:00<?, ?batch/s]c:\Users\Jean\anaconda3\envs\ActivationPrototypes_SARSeg\lib\site-packages\torch\nn\modules\module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
Epoch 1/10: 100%|██████████| 625/625 [01:21<00:00,  7.71batch/s]


Epoch 1, Loss: 3.6863, IoU: 0.1979, F1: 0.3247


Validation: 100%|██████████| 250/250 [00:23<00:00, 10.44batch/s]


...

Epoch 9, Loss: 3.6163, IoU: 0.3069, F1: 0.4647


Epoch 10/10: 100%|██████████| 625/625 [00:51<00:00, 12.17batch/s]


Epoch 10, Loss: 3.6358, IoU: 0.2766, F1: 0.4284


Validation: 100%|██████████| 250/250 [00:14<00:00, 17.17batch/s]


Epoch 10, Loss: 3.6185, IoU: 0.2987, F1: 0.4551
